<a href="https://colab.research.google.com/github/YuXuan20040221/GDpj/blob/fish/RoadDamage_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 道路安全


# *掛載 Google Drive
讓 Colab 讀取放在雲端硬碟的資料集
(這邊不用動)
/content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# *安裝相關套件

* YOLO套件

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.4 MB/s eta 0:00:00


# 設定(yaml)

In [ ]:
# 定義train.yaml ->
yaml_content = """
path: /content/drive/MyDrive/datasets  # 整體資料集根目錄
train: Choose/train_all/images # 訓練用資料夾位置
val: Choose/valid/images # 驗證用資料夾位置

nc: 1 # 類別個數
names: ["pothole"] # 類別名稱
"""

# 寫入雲端 ->
with open("/content/drive/MyDrive/datasets/trainYOLO.yaml", "w") as f:
    f.write(yaml_content)

# 訓練
參數調整參考:
[YOLO調整指南](https://docs.ultralytics.com/zh/guides/hyperparameter-tuning/)

In [ ]:
import os
import torch
import sys
import torch.nn as nn
from ultralytics import YOLO

# 建立模型

# 模型名稱: 可用YOLO原始模型、自定義模型、之前練過的
model = YOLO("/content/drive/MyDrive/model/model_Preprocessing/combined_preprocessing_yolo_run4/weights/best.pt")



# 我自己要玩的，不想用就幫我註解掉就好(刪了我會哭嗚嗚)

# 1. 自訂模型架構
# model = YOLO("/content/drive/MyDrive/model/Preprocessing.yaml")

# 2. 載入 YOLO 官方預訓練權重 (僅載入 Backbone)
# pretrained_yolo = YOLO("yolov8n.pt")
# pretrained_yolo_dict = pretrained_yolo.model.state_dict()
# model_dict = model.model.state_dict()

# # 過濾 YOLO 預訓練權重，只保留與新模型結構匹配的部分 (主要為 Backbone)
# filtered_yolo_dict = {k: v for k, v in pretrained_yolo_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
# model_dict.update(filtered_yolo_dict)
# model.model.load_state_dict(model_dict) # 不需要 strict=False 了，因為我們已經過濾和更新了字典


# # 3. 載入預訓練的 STN 權重到 Preprocessing 層的 STN 子模塊
# stn_weights_path = "/content/drive/MyDrive/model/model_Preprocessing/stn_training_output/best_stn.pth"
# if os.path.exists(stn_weights_path):
#     try:
#         stn_state_dict = YOLO(stn_weights_path)
#         # 移除 'module.' 前綴，以防是 DataParallel 訓練保存的權重
#         stn_state_dict = {k.replace('module.', ''): v for k, v in stn_state_dict.items()}

#         # 載入權重到 Preprocessing 層中的 STN 子模塊
#         # 假設 Preprocessing 層是模型的第 0 層
#         if hasattr(model.model[0], 'STN'):
#              model.model[0].STN.load_state_dict(stn_state_dict)
#              print(f"Successfully loaded STN weights from {stn_weights_path} into model.model[0].STN")
#         else:
#             print(f"Warning: model.model[0] does not have an 'STN' attribute. Could not load STN weights.")

#     except Exception as e:
#         print(f"Error loading STN weights from {stn_weights_path}: {e}")
# else:
#     print(f"Warning: STN weights file not found at {stn_weights_path}. Skipping STN weight loading.")


# 訓練模型
model.train(
    # 訓練用參數
    data="/content/drive/MyDrive/datasets/trainYOLO.yaml",
    epochs=200, # 圈數
    batch=32, # 單次處理影像數量
    imgsz=640, # 圖片大小
    project="/content/drive/MyDrive/model/model_Preprocessing", # 存在哪資料夾
    name="combined_preprocessing_yolo_run4_2", # 叫啥名
    lr0=1e-5, # 初始學習率(越小收斂速度越慢但越穩)
    lrf=0.00005, # 最終學習率(是lr0的幾%，控制下降幅度)
    weight_decay=0.0005, # L2，防overfitting(但調太大模型難學)
    optimizer="AdamW", # 優化器(AdamW好像大家都會用)
    patience=30,  # earlystopping
    seed=42,
    warmup_epochs=20, # 穩定前幾epoch
    dropout=0.4, # 關掉多少神經元
    device=0,
    # freeze=1,
    # freeze=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22],
    # resume=True, # 繼續訓練
    mosaic=0.2,
    # max_norm=5.0, # 嘗試將梯度裁剪的閾值降低
    # clip=1.0

    # 圖片處理
    hsv_h=0.015,     # 色相擾動
    hsv_s=0.7,       # 飽和度擾動
    hsv_v=0.4,       # 亮度擾動
    degrees=10.0,    # 旋轉
    translate=0.1,   # 平移
    scale=0.3,       # 縮放
    shear=5.0,       # 剪切
)

# preprocessing

In [ ]:
# from ultralytics.nn.modules.block import Preprocessing
import torchvision.utils as vutils
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super(h_sigmoid, self).__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6

# 自訂activate function
class h_tanh(nn.Module):
    def __init__(self, slope=0.2):
        super().__init__()
        self.slope = slope

    def forward(self, x):
        return torch.clamp(self.slope * x, min=-1.0, max=1.0)

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super(h_swish, self).__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)

class CoordAtt(nn.Module):
    def __init__(self, inp, oup, reduction=32):
        super(CoordAtt, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()

        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)


    def forward(self, x):
        identity = x

        n,c,h,w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()

        out = identity * a_w * a_h

        return out


class HSTN(nn.Module):
    def __init__(self, c1):
        super(HSTN, self).__init__()

        self.h_tanh = h_tanh()

        # localization network (取特徵)
        self.localization = nn.Sequential(
            nn.Conv2d(c1, 8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 12, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(12, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            CoordAtt(16, 16, 2),
            nn.AdaptiveAvgPool2d((8, 8))
        )

        self._fixed_flattened_size = 16 * 8 * 8  # C * H * W

        # 截8個點
        self.fc_loc = nn.Sequential(
            nn.Linear(self._fixed_flattened_size, 16),
            nn.ReLU(True),
            nn.Dropout(p=0.3),
            nn.Linear(16, 8),
            self.h_tanh,
        )

        nn.init.normal_(self.fc_loc[3].weight, mean=0., std=.01)

        # bias 長這樣:[
        # [1,0,0],
        # [0,1,0],
        # [0,0,1],
        #] (*最後一個1固定*)
        identity_bias = torch.tensor([4.8, 0, 0, 0, 4.8, 0, 0, 0], dtype=torch.float)
        self.fc_loc[3].bias.data.copy_(identity_bias)

    def homography_grid(self, x, theta):
      B, C, H, W = x.size()
      dtype = x.dtype
      device = x.device

      y_t, x_t = torch.meshgrid(
        torch.linspace(-1.0, 1.0, H, device=device, dtype=dtype),
        torch.linspace(-1.0, 1.0, W, device=device, dtype=dtype),
        indexing="ij"
      )

      x_t_flat = x_t.reshape(-1)  # [H*W]
      y_t_flat = y_t.reshape(-1)  # [H*W]
      ones = torch.ones_like(x_t_flat)  # [H*W]

      sampling_grid = torch.stack([x_t_flat, y_t_flat, ones], dim=0)  # [3, H*W]
      sampling_grid = sampling_grid.unsqueeze(0).repeat(B, 1, 1)      # [B, 3, H*W]

      mapped_input_hom = torch.bmm(theta, sampling_grid)  # [B, 3, H*W]

      w_coords = mapped_input_hom[:, 2:3, :]  # [B, 1, H*W]
      mapped_input_2d = mapped_input_hom[:, :2, :] / (w_coords + 1e-8)  # [B, 2, H*W]

      sampling_grid = mapped_input_2d.transpose(1, 2).reshape(B, H, W, 2)  # [B, H, W, 2]

      out = F.grid_sample(x, sampling_grid, align_corners=True)

      return out



    def forward(self, x):
        B, C, _, _ = x.size()
        dtype = x.dtype
        device = x.device

        xs = self.localization(x)
        xs = xs.view(-1, self._fixed_flattened_size)

        theta = self.fc_loc(xs)
        H = torch.cat([theta, torch.ones(B, 1, device=device, dtype=dtype)], dim=1).view(B, 3, 3) # [B, 3, 3]

        print(f"H: {H[0,:,:]}")

        H[:, 2, :2] = torch.where(H[:, 2, :2] > 0.4, 0.4, H[:, 2, :2])
        H[:, 2, :2] = torch.where(H[:, 2, :2] < -0.4, -0.4, H[:, 2, :2])

        out = self.homography_grid(x, H)

        return out, H


class Preprocessing(nn.Module):
    def __init__(self, c1, c2):
        super(Preprocessing, self).__init__()
        self.save_dir = "/content/drive/MyDrive/model/model_Preprocessing/preprocessed"
        self.STN = HSTN(c1)
        self.filter = nn.Conv2d(c1, c2, kernel_size=3, padding=1, bias=False)
        self.register_buffer("transformed_img", torch.empty(0), persistent=False)
        self.register_buffer("original_img", torch.empty(0), persistent=False)
        self.register_buffer("H", torch.empty(0), persistent=False)
        nn.init.kaiming_normal_(self.filter.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):

        y_stn, H = self.STN(x)
        output = self.filter(y_stn)
        self.transformed_img = y_stn.detach()
        self.original_img = x.detach()
        self.H = self.H.detach()

        if self.save_dir is not None and not self.training and y_stn.shape[1] == 3:
            try:
                viz_images = torch.cat([x[:4], y_stn[:4]], dim=0) # 取前4張圖，方便查看
                viz_images = (viz_images - viz_images.min()) / (viz_images.max() - viz_images.min() + 1e-6)
                vutils.save_image(
                     viz_images,
                     f"{self.save_dir}/preprocessed_batch.png",
                     nrow=4 # 每行顯示4張圖 (輸入, 輸出, 目標, ...以此類推)
                 )
            except TypeError as e:
                print(f"Warning: Could not save image due to TypeError: {e}. Data shape: {output.shape}, dtype: {output.dtype}")

        return output


## 單獨訓練STN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import torchvision.utils as vutils


# ===== 資料集 =====
class PairedDataset(Dataset):
    def __init__(self, root_a1, root_a2, transform=None):
        self.root_a1 = root_a1
        self.root_a2 = root_a2
        try:
            self.files = sorted(os.listdir(root_a1))
        except FileNotFoundError:
            print(f"Error: Input directory not found at {root_a1}")
            self.files = [] # Set files to empty list if root_a1 is not found
        self.transform = transform
        self.transform2 = transforms.Compose([
            transforms.Resize((640, 640)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_a1_path = os.path.join(self.root_a1, fname)
        img_a2_path = os.path.join(self.root_a2, fname)

        try:
            img_a1 = Image.open(img_a1_path).convert('RGB')
            img_a2 = Image.open(img_a2_path).convert('RGB')
        except FileNotFoundError:
            return None, None
        except Exception as e:
            return None, None


        if self.transform:
            img_a1 = self.transform(img_a1)
            img_a2 = self.transform2(img_a2)

        return img_a1, img_a2

def collate_fn_skip_none(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None
    return torch.utils.data.dataloader.default_collate(batch)


# ===== 訓練流程 =====
device = "cuda" if torch.cuda.is_available() else "cpu"
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
    # 影像處理
    transforms.RandomRotation(degrees=15)
])

dataset = PairedDataset("/content/drive/MyDrive/datasets/Processed/ori", "/content/drive/MyDrive/datasets/Processed/P/", transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_skip_none)

if len(dataset) == 0:
    print("Dataset is empty. Please check your dataset paths and contents.")
else:
    model = HSTN(c1=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
    criterion = F.l1_loss

    best_loss = float('inf')

    # 設定影像儲存目錄
    viz_dir = "/content/drive/MyDrive/model/model_Preprocessing/stn_training_output/stn_visualization"
    os.makedirs(viz_dir, exist_ok=True) # 創建目錄如果不存在

    torch.autograd.set_detect_anomaly(True)

    for epoch in range(100):
        total_loss = 0
        num_samples = 0

        for i, (x, y) in enumerate(dataloader):
            if x is None:
                continue

            x, y = x.to(device), y.to(device)
            y = y.detach()

            # Get the transformed image from the tuple output
            out, H = model(x)

            loss = criterion(out, y) # Calculate loss using the transformed image
            optimizer.zero_grad()
            loss.backward() # Backward pass

            optimizer.step()
            total_loss += loss.item() * x.size(0)
            num_samples += x.size(0)

            if i == 0:
                 viz_images = torch.cat([x[:4], out[:4], y[:4]], dim=0) # 取前4張圖，方便查看
                 viz_images = (viz_images - viz_images.min()) / (viz_images.max() - viz_images.min() + 1e-6)
                 vutils.save_image(
                     viz_images,
                     os.path.join(viz_dir, f"epoch_{epoch}_batch_{i}.png"),
                     nrow=4 # 每行顯示4張圖 (輸入, 輸出, 目標, ...以此類推)
                 )


        avg_loss = total_loss / num_samples if num_samples > 0 else float('inf')
        print(f"Epoch {epoch} Loss: {avg_loss:.4f}")

        if num_samples > 0 and avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "/content/drive/MyDrive/model/model_Preprocessing/stn_training_output/best_stn.pth")

    print("\nTraining finished.")
    if best_loss != float('inf'):
        print(f"Best achieved loss: {best_loss:.4f}")
    else:
        print("No valid training samples were processed.")

串流輸出內容已截斷至最後 5000 行。
        [ 0.0564,  0.0502,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.4358,  0.0663,  0.0431],
        [-0.0332,  0.0504,  0.2166],
        [-0.0039,  0.0780,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.4389,  0.1002,  0.0037],
        [-0.0078,  0.0868,  0.2201],
        [ 0.0878,  0.0511,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[0.4177, 0.0195, 0.0100],
        [0.0149, 0.1634, 0.1910],
        [0.0637, 0.0248, 1.0000]], device='cuda:0', grad_fn=<SliceBackward0>)
H: tensor([[ 0.4728, -0.0078,  0.0187],
        [ 0.0179,  0.2084,  0.2662],
        [ 0.1406,  0.0042,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.1874,  0.1266,  0.1134],
        [-0.0015, -0.2722,  0.2476],
        [ 0.1399,  0.1098,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.4042,  0.0035,  0.0035],
        [ 0.0758,  0.1915,  0.2750],
     

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import torchvision.utils as vutils
from __main__ import Preprocessing # Import Preprocessing class definition

# ===== 資料集 =====
class PairedDataset(Dataset):
    def __init__(self, root_a1, root_a2, transform=None):
        self.root_a1 = root_a1
        self.root_a2 = root_a2
        try:
            self.files = sorted(os.listdir(root_a1))
        except FileNotFoundError:
            print(f"Error: Input directory not found at {root_a1}")
            self.files = [] # Set files to empty list if root_a1 is not found
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_a1_path = os.path.join(self.root_a1, fname)
        img_a2_path = os.path.join(self.root_a2, fname)

        try:
            img_a1 = Image.open(img_a1_path).convert('RGB')
            img_a2 = Image.open(img_a2_path).convert('RGB')
        except FileNotFoundError:
            return None, None
        except Exception as e:
            return None, None


        if self.transform:
            img_a1 = self.transform(img_a1)
            img_a2 = self.transform(img_a2)

        return img_a1, img_a2

def collate_fn_skip_none(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None
    return torch.utils.data.dataloader.default_collate(batch)


# ===== 訓練流程 =====
device = "cuda" if torch.cuda.is_available() else "cpu"
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor()
])

dataset = PairedDataset("/content/drive/MyDrive/datasets/Processed/ori", "/content/drive/MyDrive/datasets/Processed/P/", transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_skip_none)

if len(dataset) == 0:
    print("Dataset is empty. Please check your dataset paths and contents.")
else:
    # Initialize the Preprocessing model with output channels matching the target image (3 channels for RGB)
    model = Preprocessing(c1=3, c2=3).to(device) # Initialize Preprocessing model with c2=3

    # Load pre-trained STN weights into the Preprocessing model's STN submodule
    stn_weights_path = "/content/drive/MyDrive/model/model_Preprocessing/stn_training_output/best_stn.pth"
    if os.path.exists(stn_weights_path):
        try:
            stn_state_dict = torch.load(stn_weights_path)
            # Remove 'module.' prefix if present
            stn_state_dict = {k.replace('module.', ''): v for k, v in stn_state_dict.items()}

            # Load weights into the STN submodule
            if hasattr(model, 'STN') and isinstance(model.STN, HSTN):
                 model.STN.load_state_dict(stn_state_dict)
                 print(f"Successfully loaded STN weights from {stn_weights_path} into Preprocessing model's STN submodule.")
            else:
                 print("Warning: Preprocessing model does not have an 'STN' attribute or it's not an HSTN instance. Skipping STN weight loading.")

        except Exception as e:
            print(f"Error loading STN weights from {stn_weights_path} into Preprocessing model: {e}")
    else:
        print(f"Warning: STN weights file not found at {stn_weights_path}. Skipping STN weight loading.")


    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5) # Increased learning rate
    criterion = F.l1_loss

    best_loss = float('inf')

    # 設定影像儲存目錄
    viz_dir = "/content/drive/MyDrive/model/model_Preprocessing/stn_training_output/stn_visualization"
    os.makedirs(viz_dir, exist_ok=True) # 創建目錄如果不存在

    # Enable anomaly detection to pinpoint the inplace operation
    torch.autograd.set_detect_anomaly(True)

    # Train for a few epochs to integrate the STN weights
    num_epochs_short_train = 5 # Train for 5 epochs
    output_model_path = "/content/drive/MyDrive/model/model_Preprocessing/preprocessing_trained.pth"


    for epoch in range(num_epochs_short_train):
        total_loss = 0
        num_samples = 0
        model.train() # Set model to training mode

        for i, (x, y) in enumerate(dataloader):
            if x is None:
                continue

            x, y = x.to(device), y.to(device)
            y = y.detach()

            # Get the transformed image and H from the tuple output
            out = model(x)


            loss = criterion(out, y) # Calculate loss using the transformed image
            optimizer.zero_grad()
            loss.backward() # Backward pass

            # Add gradient clipping here if needed during this short training
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Example clipping


            optimizer.step()
            total_loss += loss.item() * x.size(0)
            num_samples += x.size(0)

            # Save visualization only for the first batch of the first epoch
            if epoch == 0 and i == 0:
                 model.eval() # Set model to evaluation mode for consistent visualization
                 with torch.no_grad():
                     # Get output and H for visualization
                     viz_out = model(x[:4].to(device))
                     viz_images = torch.cat([x[:4].cpu(), viz_out[:4].cpu(), y[:4].cpu()], dim=0) # 取前4張圖，方便查看
                     viz_images = (viz_images - viz_images.min()) / (viz_images.max() - viz_images.min() + 1e-6)
                     vutils.save_image(
                         viz_images,
                         os.path.join(viz_dir, f"preprocessing_short_train_epoch_{epoch}_batch_{i}.png"),
                         nrow=4 # 每行顯示4張圖 (輸入, 輸出, 目標, ...以此類推)
                     )
                 model.train() # Set model back to training mode


        avg_loss = total_loss / num_samples if num_samples > 0 else float('inf')
        print(f"Preprocessing Short Train Epoch {epoch} Loss: {avg_loss:.4f}")

        # Save the model state dict after each epoch (optional, can save best loss too)
        torch.save(model.state_dict(), output_model_path)
        print(f"Saved Preprocessing model state dict to {output_model_path}")


    print("\nShort training of Preprocessing model finished.")
    print(f"You can now try loading the state dict from {output_model_path} into your YOLO model.")

Successfully loaded STN weights from /content/drive/MyDrive/model/model_Preprocessing/stn_training_output/best_stn.pth into Preprocessing model's STN submodule.
H: tensor([[ 0.5061,  0.0240,  0.0510],
        [ 0.0185,  0.1598,  0.1818],
        [ 0.2037, -0.2119,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.5148,  0.0252,  0.0256],
        [ 0.0147,  0.1635,  0.1831],
        [ 0.1479, -0.2288,  1.0000]], device='cuda:0')
H: tensor([[ 0.6409,  0.0247,  0.0483],
        [-0.0075,  0.3074,  0.1529],
        [ 0.0364, -0.1565,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 5.4861e-01,  2.0828e-04,  3.2908e-02],
        [-5.3126e-03,  7.5923e-02,  2.1961e-01],
        [ 1.3570e-01, -2.3473e-01,  1.0000e+00]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([[ 0.6134,  0.0041,  0.0260],
        [ 0.0180,  0.3263,  0.1576],
        [ 0.2024, -0.1865,  1.0000]], device='cuda:0',
       grad_fn=<SliceBackward0>)
H: tensor([

# 模型評估

## 效能

In [ ]:
from ultralytics import YOLO
import os
import torch
import torchvision.utils as vutils

def save_preprocessed_images(images, save_dir="runs/preprocessed", step=0):
    # images: Tensor [B, C, H, W]
    vutils.save_image(images, f"{save_dir}/batch_{step}.png", nrow=4, normalize=True)


# 載入訓練好的模型 (請確認路徑正確)
# 你可以使用 best.pt 或 last.pt
model_path = "/content/drive/MyDrive/model/model_Preprocessing/preprocessing_yolo_run_L1S1/weights/best.pt"
model = YOLO(model_path)

# 定義測試資料集的路徑 (請確認路徑和 trainYOLO.yaml 中的 val 路徑一致)
# 如果你的測試資料集是獨立的，需要另外建立一個 yaml 檔案來指定測試資料夾
# 這裡先使用 trainYOLO.yaml 中的 val 路徑作為示範
data_yaml_path = "/content/drive/MyDrive/datasets/trainYOLO.yaml"


# 在測試資料集上進行驗證
# 你可以使用 split='test' 如果你的 yaml 檔案中有定義測試集
# 這裡使用 split='val' 因為 trainYOLO.yaml 中只有 train 和 val
results = model.val(data=data_yaml_path, split='val')

# 打印評估結果
print("\n--- 模型評估結果 ---")
print(f"mAP50-95: {results.box.map}")
print(f"mAP50: {results.box.map50}")
print(f"Precision: {results.box.mp}")
print(f"Recall: {results.box.mr}")
print(f"F1 Score: {(2 * results.box.mp * results.box.mr) / (results.box.mp + results.box.mr) if (results.box.mp + results.box.mr) > 0 else 0}") # 手動計算 F1 Score

# 打印模型性能資訊
print("\n--- 模型性能資訊 ---")
# 模型參數數量
# Directly calculate trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型可訓練參數數量 (trainable parameters): {trainable_params}")

# Total model parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"模型總參數數量 (total parameters): {total_params}")


# GFLOPs (在 val 時會計算並顯示在上面的輸出中)
print("GFLOPs (請參考上方 val 執行輸出的 summary 部分)")

# 模型大小 (通過文件大小估計)
model_file_size = os.path.getsize(model_path) / (1024 * 1024) # 轉為 MB
print(f"模型檔案大小: {model_file_size:.2f} MB")

print("\n--- 評估結束 ---")


Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 53.2±7.3 MB/s, size: 123.4 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1QqP8GboDMEQLNFct8SmVahrhoUgJEXjC/datasets/valid/labels.cache... 85 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 85/85 113.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.4s
                   all         85        343      0.515      0.309      0.343      0.135
Speed: 5.7ms preprocess, 4.4ms inference, 0.0ms loss, 5.0ms postprocess per image
Results saved to /content/runs/detect/val5

--- 模型評估結果 ---
mAP50-95: 0.1351032502248996
mAP50: 0.3432500698716208
Precision: 0.5148896989248312
Recall: 0.30903790087463556
F1 Score: 0.3862485776090782

--- 模型性能資訊 ---
模型可訓練參數數量 (trainable parameters): 0
模型總參數數量 (tot

In [ ]:
!pip uninstall torch torchvision sympy -y
!pip install torch torchvision sympy

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: sympy 1.13.3
Uninstalling sympy-1.13.3:
  Successfully uninstalled sympy-1.13.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.7 MB/s

# 模組化程式碼

## 1. 定義 CombinedModel 結構

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO # 引入 Ultralytics 的 YOLO 類別來載入模型結構
import types # 引入 types 模組，用於動態修改方法

# 確保你能從 __main__ 載入你的 Preprocessing 模型定義
# 這裡假設你的 Preprocessing 和 HSTN 類別已經在前面的儲存格中定義並執行過
from __main__ import Preprocessing, HSTN

# --- 定義 CombinedModel --- (這個類別我們將不再需要)
# class CombinedModel(nn.Module):
#     ... # 移除 CombinedModel 類別定義

# --- 實例化模型並載入權重 ---

# 將模型實例化和權重載入的程式碼放在這裡

# 實例化你的預處理模型
# 確保 c1 匹配輸入圖片的通道數 (例如，RGB 圖片是 3 個通道)
# c2 應該匹配 Preprocessing 模組的輸出通道數
preprocessing_model = None
try:
    preprocessing_model = Preprocessing(c1=3, c2=3) # 假設 Preprocessing 輸出 3 個通道
    print("Preprocessing 模型實例化成功。")
except NameError:
    print("錯誤：Preprocessing 類別未定義。請確保前面儲存格已成功執行。")
except Exception as e:
    print(f"實例化 Preprocessing 模型時發生錯誤：{e}")
    preprocessing_model = None


# 載入已經訓練好的預處理模型權重
preprocessing_weights_path = "/content/drive/MyDrive/model/model_Preprocessing/preprocessing_trained.pth"
if preprocessing_model is not None and os.path.exists(preprocessing_weights_path):
    try:
        preprocessing_state_dict = torch.load(preprocessing_weights_path, map_location='cpu') # 載入權重，先載入到 CPU 避免 GPU 記憶體問題
        # 移除 'module.' 前綴，以防是 DataParallel 和 AMP 保存的權重
        preprocessing_state_dict = {k.replace('module.', '').replace('_orig_mod.', ''): v for k, v in preprocessing_state_dict.items()}
        # 載入權重到預處理模型
        preprocessing_model.load_state_dict(preprocessing_state_dict)
        print(f"成功從 {preprocessing_weights_path} 載入 Preprocessing 權重。")
    except Exception as e:
        print(f"載入 Preprocessing 權重時發生錯誤：{e}")
        preprocessing_model = None # 載入失敗則設為 None
else:
    if preprocessing_model is not None:
         print(f"警告：找不到 Preprocessing 權重檔案 {preprocessing_weights_path}。跳過載入。")
    # 如果模型實例化失敗，這裡也會跳過


# 創建 YOLO 物件並載入權重 (使用 YOLO() 方法)
# 這個 YOLO 物件將用於啟動訓練
yolo_trained_model_path = "/content/drive/MyDrive/model/model_n/yolov8n_Processed_best/weights/best.pt" # 你的 YOLO 訓練好的模型檔案路徑 (.pt)
trainer = None # 我們將使用這個變數來指向 YOLO 物件

if os.path.exists(yolo_trained_model_path):
    try:
        # 使用 YOLO() 創建 YOLO 物件，它會自動載入模型結構和權重
        trainer = YOLO(yolo_trained_model_path)
        print(f"成功創建 YOLO 物件並從 {yolo_trained_model_path} 載入模型。")

        # 獲取 YOLO 物件內部的 DetectionModel 實例
        yolo_model_instance = trainer.model
        print("成功獲取 YOLO 物件內部的 DetectionModel 實例。")

    except Exception as e:
        print(f"創建 YOLO 物件或載入模型時發生錯誤：{e}")
        trainer = None # 載入失敗則設為 None
        yolo_model_instance = None

else:
    print(f"警告：找不到 YOLO 訓練好的模型檔案 {yolo_trained_model_path}。無法創建 YOLO 物件。")
    trainer = None
    yolo_model_instance = None


# --- 動態整合 Preprocessing 模組到 YOLO 模型實例 ---
# 只有當 Preprocessing 模型和 YOLO 模型實例都成功載入時才進行整合
if preprocessing_model is not None and yolo_model_instance is not None:
    print("\n開始整合 Preprocessing 模組到 YOLO 模型實例...")

    # 1. 將 Preprocessing 模型添加為 YOLO 模型實例的一個子模組
    # 這樣 PyTorch 才能正確管理 Preprocessing 模組的參數和梯度
    # 我們給它一個名字，例如 'preprocessing_layer'
    # 注意：模組名稱不能和 YOLO 模型內部已有的層名稱衝突
    yolo_model_instance.preprocessing_layer = preprocessing_model
    print("成功將 Preprocessing 模型添加為 YOLO 模型實例的子模組。")

    # 2. 動態修改 YOLO 模型實例的 forward 方法
    # 定義一個新的 forward 函數，它將替代 DetectionModel 實例原有的 forward 方法
    # 這個新的 forward 函數需要能訪問到 yolo_model_instance (self)
    # 注意：這裡我們修改的是 yolo_model_instance 這個物件的 forward 方法

    # 保存原有的 YOLO forward 方法，以便在新方法中調用
    original_yolo_forward = yolo_model_instance.forward

    # 定義新的 forward 方法
    # 這個方法會接收 Ultralytics Trainer 傳入的標準訓練批次字典 (訓練模式) 或圖片 Tensor (推論模式)
    def new_yolo_forward(self, x, *args, **kwargs):
        # 確保模型在正確的設備上
        device = next(self.parameters()).device # 獲取模型所在的設備

        # 將 Preprocessing 模組移動到正確的設備 (如果還沒有)
        self.preprocessing_layer.to(device)

        # 判斷當前模式 (訓練或推論)
        if isinstance(x, dict): # Trainer 在訓練時傳入字典 {'img': ..., 'instances': ...}
             # 訓練模式
             batch_dict = x # 接收 Trainer 傳入的整個批次字典
             original_images = batch_dict['img'].to(device) # 原始圖片 Tensor
             targets = batch_dict.get('instances', None) # 標籤 Tensor (可能為 None)

             # 將原始圖片傳入 Preprocessing 模組
             processed_img = self.preprocessing_layer(original_images)

             # 創建一個新的批次字典給原有的 YOLO forward/loss 使用
             # 用處理後的圖片替換原來的圖片
             # 確保 targets 也在正確的設備上
             # 注意：這裡我們不再需要像之前那樣拆分 targets 了，直接傳入原始 targets Tensor 即可
             # YOLO 內部 Loss 層會處理 targets 的格式
             yolo_batch_dict = {'img': processed_img, 'instances': targets.to(device) if targets is not None else None}

             # 呼叫原有的 YOLO forward 方法，傳入新的批次字典
             # original_yolo_forward 在訓練時接收字典輸入，會自動計算 YOLO Loss 並回傳損失組件 (例如 tuple)
             # 我們決定只用 YOLO Loss，所以直接返回 YOLO 內部計算的損失組件即可
             yolo_loss_output = original_yolo_forward(yolo_batch_dict)

             # 返回 YOLO 內部計算的損失組件 tuple
             # Ultralytics Trainer 期望 forward 在訓練時回傳 losses (通常是包含 box, cls, dfl losses 的 tuple)
             return yolo_loss_output # 直接返回 YOLO 內部計算的損失組件

        else: # 推論或驗證模式 (只傳入圖片 Tensor)
            # 將輸入圖片傳入 Preprocessing 模組
            # 確保輸入圖片 x 在正確的設備上
            input_img = x.to(device)
            processed_img = self.preprocessing_layer(input_img)

            # 將處理後的圖片傳入 YOLO 模型原有的 forward 方法
            # original_yolo_forward 在推論時接收圖片 Tensor，回傳預測結果
            # 明確傳入 profile=False，避免多值 Tensor 被誤解
            yolo_predictions = original_yolo_forward(processed_img, profile=False, *args, **kwargs) # 傳遞其他可能的 args/kwargs

            # 在推論/驗證模式下，只返回 YOLO 的原始預測結果
            return yolo_predictions


    # 將新的 forward 方法綁定到 YOLO 模型實例上
    # 使用 types.MethodType 將函數綁定為實例方法
    yolo_model_instance.forward = types.MethodType(new_yolo_forward, yolo_model_instance)
    print("成功修改 YOLO 模型實例的 forward 方法。")

    # 將模型移動到可用的設備 (GPU 或 CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    yolo_model_instance.to(device) # 將修改後的模型實例移動到設備
    print(f"將修改後的模型實例移動到設備: {device}")


else:
    print("\n無法進行模型整合，因為預處理模型或 YOLO 模型載入失敗。")
    # 如果整合失敗，trainer 變數可能已經是 None


# 現在 trainer 變數是一個 YOLO 物件，其內部的模型已經被修改
# 你可以直接使用 trainer.train(...) 來啟動訓練了
print("\n模型定義、載入和整合程式碼更新完畢。")
print("trainer 變數現在是一個 YOLO 物件，其內部模型包含了 Preprocessing 模組。")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Preprocessing 模型實例化成功。
成功從 /content/drive/MyDrive/model/model_Preprocessing/preprocessing_trained.pth 載入 Preprocessing 權重。
成功創建 YOLO 物件並從 /content/drive/MyDrive/model/model_n/yolov8n_Processed_best/weights/best.pt 載入模型。
成功獲取 YOLO 物件內部的 DetectionModel 實例。

開始整合 Preprocessing 模組到 YOLO 模型實例...
成功將 Preprocessing 模型添加為 YOLO 模型實例的子模組。
成功修改 YOLO 模型實例的 forward 方法。
將修改後的模型實例移動到設備: cuda

模型定義、載入和整合程式碼更新完畢。
trainer 變數現在是一個 YOLO 物件，其內部模型包含了 Preprocessing 模組。


### 啟動訓練

現在我們已經成功載入並整合了 Preprocessing 模組到 YOLO 模型中，並且將修改後的模型賦值給 `trainer` 變數。

你可以使用 `trainer.train()` 方法來啟動 Ultralytics 標準的訓練流程了。請根據你的訓練需求設定以下參數：

In [ ]:
# 啟動訓練
# trainer 變數是一個 YOLO 物件，其內部模型已經包含了 Preprocessing 模組

if trainer is not None:
    try:
        print("開始使用 trainer.train() 啟動訓練...")
        print(f"preprocessing_layer: {hasattr(trainer.model, 'preprocessing_layer')}")
        # 確保使用正確的設備進行訓練
        # 如果 CombinedModel 已經被移動到設備，這裡的 device 參數可以省略或與之匹配
        # Ultralytics Trainer 通常會自己處理設備分配，但明確指定更保險
        training_device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"訓練將在設備 {training_device} 上運行。")


        # --- 設定你的訓練參數 ---
        # 請替換成你的實際資料設定檔路徑或參數
        data_config_path = "/content/drive/MyDrive/datasets/trainYOLO.yaml" # 替換成你的資料設定檔路徑

        # 這裡直接使用 data_config_path 來啟動訓練
        # Ultralytics Trainer 會根據這個設定檔創建 DataLoader
        # 如果你想使用之前手動創建的 DataLoader，可能需要修改 Ultralytics Trainer 的原始碼或尋找相關選項 (通常比較複雜)
        # 因此，建議直接提供 data_config_path
        if os.path.exists(data_config_path):
            print(f"使用資料設定檔: {data_config_path}")
            train_results = trainer.train(
                data=data_config_path,
                epochs=300, # 總 epoch 數
                batch=32,    # 批次大小 (請根據你的 GPU 記憶體調整)
                imgsz=640,  # 圖片尺寸
                workers=4,  # DataLoader 工作進程數
                name='combined_preprocessing_yolo_run_2', # 訓練名稱
                cache=False,
                device=training_device, # 指定訓練設備
                project="/content/drive/MyDrive/model/model_Preprocessing", # 存在哪資料夾
                lr0=1e-4, # 初始學習率(越小收斂速度越慢但越穩)
                lrf=0.0005, # 最終學習率(是lr0的幾%，控制下降幅度)
                weight_decay=0.0005, # L2，防overfitting(但調太大模型難學)
                optimizer="AdamW", # 優化器(AdamW好像大家都會用)
                patience=50,  # earlystopping
                seed=42,
                warmup_epochs=20, # 穩定前幾epoch
                dropout=0.2, # 關掉多少神經元
                # resume=True, # 繼續訓練  等等加:
                box=0.5,
                cls=0.5,
                # anchor_free=True,
                mosaic=0.2,

                    # 圖片處理
                hsv_h=0.015,     # 色相擾動
                hsv_s=0.01,       # 飽和度擾動
                hsv_v=0.2,       # 亮度擾動
                degrees=15.0,    # 旋轉
                translate=0.1,   # 平移
                scale=0.3,       # 縮放
                shear=2.0,       # 剪切
            )
            print("\n訓練完成。")
            # train_results 包含了訓練過程的結果，例如最佳權重路徑等

        else:
            print(f"錯誤：找不到資料設定檔 {data_config_path}。請檢查路徑。")


    except Exception as e:
        print(f"\n啟動訓練時發生錯誤：{e}")

else:
    print("\ntrainer 物件未成功創建，無法啟動訓練。請檢查前面的儲存格。")

開始使用 trainer.train() 啟動訓練...
preprocessing_layer: True
訓練將在設備 cuda 上運行。
使用資料設定檔: /content/drive/MyDrive/datasets/trainYOLO.yaml
Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=0.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/datasets/trainYOLO.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.01, hsv_v=0.2, imgsz=720, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.0005, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/model/model_n/yolov8n_Process